In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 데이터셋 로딩, RoBERTa 학습, 성능 평가에 필요한 라이브러리를 설치합니다.
!pip -q install transformers datasets accelerate scikit-learn

In [ ]:
# 데이터 처리, 시각화, PyTorch 학습에 사용할 모듈을 불러옵니다.
import re
import html
import random
import hashlib

import numpy as np
import pandas as pd
import torch

from IPython.display import display, HTML
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from torch.utils.data import TensorDataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

In [ ]:
# 사용 가능한 GPU 또는 다른 장치를 선택해 모델과 입력 데이터를 같은 장치에 올립니다.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    DEVICE = torch.device("xpu")
else:
    DEVICE = torch.device("cpu")

print("DEVICE:", DEVICE)

In [ ]:
# 사전학습 RoBERTa의 Tokenizer를 불러와 글을 모델 입력으로 변환할 준비를 합니다.
MODEL_NAME = "klue/roberta-base"  # Hugging Face 모델 ID
MAX_LENGTH = 256                  # 한 번에 모델에 넣을 최대 토큰 수

# 모델과 함께 공개된 Tokenizer를 사용해야 토큰 ID가 맞음
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("모델:", MODEL_NAME)
print("Tokenizer:", type(tokenizer).__name__)

In [ ]:
# 셀 생략 가능
# 짧은 예시 문장을 토큰과 숫자 ID로 바꾸어 Tokenizer 동작을 확인합니다.

sample_text = "이 글이 사람의 답변인지 ChatGPT의 답변인지 판별합니다."

tokens = tokenizer.tokenize(sample_text)   # 글을 토큰으로 분리
token_ids = tokenizer.encode(sample_text)  # 토큰을 숫자 ID로 변환

print("원문:", sample_text)
print("토큰:", tokens)
print("토큰 ID:", token_ids)

In [ ]:
# 셀 생략 가능
# 예시 문장의 input_ids와 attention_mask 모양을 확인합니다.
# 이 셀을 실행하려면 셀 6의 sample_text가 필요함

sample_encoded = tokenizer(
    [sample_text, "두 번째 예시 문장입니다."],
    padding=True,          # 두 문장의 길이를 맞춤
    truncation=True,       # 최대 길이를 넘으면 자름
    max_length=MAX_LENGTH,
    return_tensors="pt",   # PyTorch Tensor로 반환
)

print("input_ids:", sample_encoded["input_ids"].shape)
print("attention_mask:", sample_encoded["attention_mask"].shape)

print("\ninput_ids:")       # 토큰의 숫자 ID
print(sample_encoded["input_ids"])

print("\nattention_mask:")  # 실제 토큰 1, padding 0
print(sample_encoded["attention_mask"])

In [ ]:
# Hugging Face Hub에서 한국어 HC3-ko 데이터셋을 불러옵니다.
DATASET_NAME = "nayohan/HC3-ko"  # Hugging Face 데이터셋 ID

# 데이터셋의 train 분할을 Hugging Face Hub에서 불러옴
raw_dataset = load_dataset(
    DATASET_NAME,
    split="train",
)

print(raw_dataset)
print("전체 질문 행 수:", len(raw_dataset))

In [ ]:
# 셀 생략 가능
# 원본 컬럼과 예시 값을 확인해 전처리에 필요한 데이터 형태를 살펴봅니다.

print("컬럼:", raw_dataset.column_names)

# 실제 값이 어떤 모양인지 앞부분만 확인
sample_row = raw_dataset[0]

for key, value in sample_row.items():
    print(f"\n[{key}]")
    print(str(value)[:300])

In [ ]:
# 답변 목록이 문자열이나 리스트로 들어온 경우를 모두 처리해 실제 답변만 추출합니다.
def parse_answers(value):
    # 데이터가 이미 목록으로 읽힌 경우
    if isinstance(value, list):
        answers = value

    elif isinstance(value, str):
        value = value.strip()

        if not value:
            return []

        if value.startswith("["):
            # "['답변1', '답변2']"처럼 저장된 문자열을 목록으로 변환
            try:
                answers = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return []
        else:
            # 일반 문자열이면 답변 하나로 처리
            answers = [value]

    else:
        return []

    if not isinstance(answers, list):
        return []

    # 빈 답변과 문자열이 아닌 값을 제외
    return [
        answer.strip()
        for answer in answers
        if isinstance(answer, str) and answer.strip()
    ]

In [ ]:
# 같은 질문에서 사람 답변과 ChatGPT 답변을 한 개씩 골라 비교 가능한 쌍을 만듭니다.
pairs = []

for row_number, row in enumerate(raw_dataset):
    human_answers = parse_answers(
        row["human_answers"]
    )

    chatgpt_answers = parse_answers(
        row["chatgpt_answers"]
    )

    # 사람과 ChatGPT 답변이 모두 있는 질문만 사용
    if not human_answers or not chatgpt_answers:
        continue

    # 질문당 각 클래스에서 첫 번째 답변 하나씩 선택
    pairs.append({
        "question": str(row["question"]).strip(),
        "human_text": human_answers[0],
        "chatgpt_text": chatgpt_answers[0],
        "source": row["source"],
        "row_number": row_number,
    })

pairs_df = pd.DataFrame(pairs)

print("사용 가능한 질문·답변 쌍:", len(pairs_df))

In [ ]:
# Colab 학습 시간을 조절하기 위해 사용할 질문 쌍의 수를 제한합니다.
MAX_PAIRS = 4000  # Colab 학습 시간을 고려해 사용할 질문 수를 제한

if len(pairs_df) > MAX_PAIRS:
    pairs_df = pairs_df.sample(
        n=MAX_PAIRS,
        random_state=SEED,
    ).reset_index(drop=True)

print("사용 질문 수:", len(pairs_df))
print("예상 답변 수:", len(pairs_df) * 2)

In [ ]:
# 셀 생략 가능
# 사람 답변과 ChatGPT 답변의 글 길이 분포를 비교합니다.

# 두 클래스의 글 길이 차이를 확인하는 분석용 셀
pairs_df["human_length"] = pairs_df["human_text"].str.len()
pairs_df["chatgpt_length"] = pairs_df["chatgpt_text"].str.len()

display(
    pairs_df[
        ["human_length", "chatgpt_length"]
    ].describe()
)

In [ ]:
# 같은 질문이 학습과 평가에 섞이지 않도록 질문 단위로 학습 데이터를 나눕니다.
# 같은 질문이 학습·평가 데이터 양쪽에 들어가지 않도록 그룹으로 분할
splitter_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=SEED,
)

train_idx, temp_idx = next(
    splitter_1.split(
        pairs_df,
        groups=pairs_df["question"],
    )
)

train_pairs = pairs_df.iloc[train_idx].reset_index(drop=True)
temp_pairs = pairs_df.iloc[temp_idx].reset_index(drop=True)

print("학습 질문:", len(train_pairs))
print("임시 질문:", len(temp_pairs))

In [ ]:
# 남은 질문을 검증용과 시험용으로 나누어 모델 선택과 최종 평가를 분리합니다.
# 남은 질문을 검증용과 시험용으로 절반씩 나눔
splitter_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.5,
    random_state=SEED,
)

val_idx, test_idx = next(
    splitter_2.split(
        temp_pairs,
        groups=temp_pairs["question"],
    )
)

val_pairs = temp_pairs.iloc[val_idx].reset_index(drop=True)
test_pairs = temp_pairs.iloc[test_idx].reset_index(drop=True)

print("검증 질문:", len(val_pairs))
print("시험 질문:", len(test_pairs))

In [ ]:
# 세 분할에 같은 질문이 중복되지 않는지 검사해 데이터 누출을 막습니다.
train_questions = set(train_pairs["question"])
val_questions = set(val_pairs["question"])
test_questions = set(test_pairs["question"])

# 중복 질문이 있으면 데이터 누출 가능성이 있으므로 실행 중지
assert train_questions.isdisjoint(val_questions)
assert train_questions.isdisjoint(test_questions)
assert val_questions.isdisjoint(test_questions)

print("분할 사이에 같은 질문이 없습니다.")

In [ ]:
# 답변 쌍을 개별 글과 사람(0)·ChatGPT(1) 라벨로 바꿉니다.
def make_classification_df(pair_df):
    rows = []

    for _, row in pair_df.iterrows():
        # 사람 답변: label 0
        rows.append({
            "text": row["human_text"],
            "label": 0,
            "source": row["source"],
        })

        # ChatGPT 답변: label 1
        rows.append({
            "text": row["chatgpt_text"],
            "label": 1,
            "source": row["source"],
        })

    return pd.DataFrame(rows)


# 질문을 먼저 분할한 뒤 답변을 펼쳐 관련 답변이 섞이지 않게 함
train_df = make_classification_df(train_pairs)
val_df = make_classification_df(val_pairs)
test_df = make_classification_df(test_pairs)

print("학습 글:", len(train_df))
print("검증 글:", len(val_df))
print("시험 글:", len(test_df))

In [ ]:
# 셀 생략 가능
# 학습·검증·시험 데이터에 두 클래스가 각각 얼마나 있는지 확인합니다.

# 각 분할에서 사람·ChatGPT 데이터 수를 확인
for name, part in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n{name}")
    print(part["label"].value_counts().sort_index())

In [ ]:
# 글을 토큰화하고 정답과 함께 반환하는 PyTorch Dataset을 정의합니다.
class TextClassificationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()

        # 글을 RoBERTa가 읽을 수 있는 토큰 ID로 미리 변환
        self.encoded = tokenizer(
            self.texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

    def __len__(self):
        # Dataset에 들어 있는 글의 개수
        return len(self.labels)

    def __getitem__(self, index):
        # DataLoader에 입력과 정답을 한 건씩 전달
        return {
            "input_ids": self.encoded["input_ids"][index],
            "attention_mask": self.encoded["attention_mask"][index],
            "labels": torch.tensor(
                self.labels[index],
                dtype=torch.long,
            ),
        }

In [ ]:
# 학습·검증·시험용 Dataset에 동일한 토큰화 기준을 적용합니다.
# 학습·검증·시험에 동일한 Tokenizer와 길이 기준 사용
train_dataset = TextClassificationDataset(
    train_df, tokenizer, MAX_LENGTH
)

val_dataset = TextClassificationDataset(
    val_df, tokenizer, MAX_LENGTH
)

test_dataset = TextClassificationDataset(
    test_df, tokenizer, MAX_LENGTH
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
# Dataset을 배치로 묶어 모델 학습과 평가에 순서대로 전달합니다.
BATCH_SIZE = 16
NUM_WORKERS = 0  # 노트북 실행을 단순하게 유지

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,   # 학습할 때는 글 순서를 섞음
    num_workers=NUM_WORKERS,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # 평가할 때는 순서를 유지
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# 셀 생략 가능
# 한 배치의 입력과 라벨 형태가 모델이 기대하는 크기인지 확인합니다.

# 모델에 들어갈 한 묶음의 크기를 확인
batch = next(iter(train_loader))

print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["labels"].shape)
print("예시 labels:", batch["labels"][:8])

In [ ]:
# 사전학습 RoBERTa에 2개 클래스용 분류 층을 붙여 모델을 준비합니다.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,    # 한국어 사전학습 RoBERTa 가져오기
    num_labels=2,  # 사람·ChatGPT 두 클래스로 분류
    id2label={
        0: "HUMAN",
        1: "CHATGPT",
    },
    label2id={
        "HUMAN": 0,
        "CHATGPT": 1,
    },
)

model = model.to(DEVICE)  # 모델을 GPU 또는 CPU로 이동

print(model.config.id2label)

In [ ]:
# 셀 생략 가능
# 예시 배치를 통과시켜 모델이 글마다 2개 클래스 점수를 내는지 확인합니다.
# 이 셀을 실행하려면 셀 22의 batch가 필요함

sample_input = {
    "input_ids": batch["input_ids"][:2].to(DEVICE),
    "attention_mask": batch["attention_mask"][:2].to(DEVICE),
}

model.eval()  # 학습 없이 출력 모양만 확인

with torch.no_grad():
    sample_output = model(**sample_input)

# 글 2개, 클래스 2개이므로 (2, 2)가 예상됨
print("logits shape:", sample_output.logits.shape)
print("logits:", sample_output.logits)

In [ ]:
# Optimizer와 학습률 Scheduler를 설정해 Fine-tuning을 준비합니다.
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

total_steps = len(train_loader) * EPOCHS

# 학습 초반 안정화 후 학습률을 점차 낮춤
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps,
)

print("Epochs:", EPOCHS)
print("Learning rate:", LR)
print("Total steps:", total_steps)

In [ ]:
# GPU에서는 혼합 정밀도를 사용해 메모리 사용량과 학습 시간을 줄입니다.
USE_AMP = DEVICE.type == "cuda"  # GPU에서만 혼합 정밀도 사용

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

print("AMP enabled:", USE_AMP)

In [ ]:
# 평가 모드에서 손실과 Accuracy·Precision·Recall·F1·AUC를 계산합니다.
@torch.no_grad()  # 평가 중에는 기울기 계산이 필요 없음
def evaluate(model, loader, device):
    model.eval()

    total_loss = 0.0
    all_labels = []
    all_predictions = []
    all_scores = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # 정답을 넣어 손실값과 분류 점수를 함께 계산
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        # 클래스 1(ChatGPT)의 모델 출력 점수
        probabilities = torch.softmax(
            outputs.logits,
            dim=1,
        )[:, 1]

        # 두 클래스 중 점수가 높은 쪽을 선택
        predictions = outputs.logits.argmax(dim=1)

        total_loss += outputs.loss.item()

        all_labels.extend(labels.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())
        all_scores.extend(probabilities.cpu().tolist())

    # 손실과 분류 성능 지표 반환
    return {
        "loss": total_loss / len(loader),
        "accuracy": accuracy_score(
            all_labels, all_predictions
        ),
        "precision": precision_score(
            all_labels, all_predictions, zero_division=0
        ),
        "recall": recall_score(
            all_labels, all_predictions, zero_division=0
        ),
        "f1": f1_score(
            all_labels, all_predictions, zero_division=0
        ),
        "auc": roc_auc_score(
            all_labels, all_scores
        ),
        "labels": all_labels,
        "predictions": all_predictions,
        "scores": all_scores,
    }

In [ ]:
# 한 Epoch 동안 배치별 역전파와 가중치 갱신을 수행합니다.
def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    scaler,
    device,
):
    model.train()  # 학습 모드 활성화

    total_loss = 0.0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        # GPU에서는 AMP를 사용해 메모리 사용과 연산 비용을 줄임
        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP,
        ):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        # 지나치게 큰 기울기를 제한
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# 최적 모델과 학습 결과를 저장할 Google Drive 경로를 준비합니다.
MODEL_DIR = Path(
    "/content/drive/MyDrive/HC3_ko_RoBERTa_results"
)

BEST_MODEL_DIR = MODEL_DIR / "best_model"

# 모델과 평가 결과를 저장할 폴더 생성
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("결과 저장 위치:", MODEL_DIR)

In [ ]:
# 여러 Epoch를 학습하고 검증 F1이 가장 높은 모델을 저장합니다.
history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_f1": [],
    "val_auc": [],
}

best_f1 = -1.0  # 지금까지 가장 높은 검증 F1

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        scaler,
        DEVICE,
    )

    val_result = evaluate(
        model,
        val_loader,
        DEVICE,
    )

    # Epoch별 결과를 저장해 나중에 표와 그래프로 확인
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_result["loss"])
    history["val_accuracy"].append(val_result["accuracy"])
    history["val_f1"].append(val_result["f1"])
    history["val_auc"].append(val_result["auc"])

    elapsed = time.time() - start

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_result['loss']:.4f} | "
        f"Val Acc: {val_result['accuracy']:.4f} | "
        f"Val F1: {val_result['f1']:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if val_result["f1"] > best_f1:
        # 검증 성능이 좋아졌을 때만 모델을 저장
        best_f1 = val_result["f1"]

        model.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)

        print("  → 최적 모델 저장")

In [ ]:
# Epoch별 기록을 표로 정리해 학습 경향 확인과 CSV 저장에 사용합니다.
# 이후 그래프와 CSV 저장에도 사용하므로 실행 필요
history_df = pd.DataFrame(history)

history_df.insert(
    0,
    "epoch",
    range(1, len(history_df) + 1),
)

display(history_df)

In [ ]:
# 셀 생략 가능
# 학습 손실과 검증 손실을 그래프로 비교해 학습 상태를 살펴봅니다.

epochs = history_df["epoch"]

plt.figure(figsize=(7, 4))

plt.plot(
    epochs,
    history_df["train_loss"],
    marker="o",
    label="Train Loss",
)

plt.plot(
    epochs,
    history_df["val_loss"],
    marker="o",
    label="Validation Loss",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training History")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 셀 생략 가능
# 검증 Accuracy와 F1 변화를 그래프로 확인합니다.

plt.figure(figsize=(7, 4))

plt.plot(
    history_df["epoch"],
    history_df["val_accuracy"],
    marker="o",
    label="Validation Accuracy",
)

plt.plot(
    history_df["epoch"],
    history_df["val_f1"],
    marker="o",
    label="Validation F1",
)

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.title("Validation Performance")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 마지막 모델 대신 검증 F1이 가장 높았던 저장 모델을 불러옵니다.
# 마지막 Epoch가 아닌 검증 F1이 가장 높았던 모델 사용
best_model = (
    AutoModelForSequenceClassification
    .from_pretrained(BEST_MODEL_DIR)
    .to(DEVICE)
)

best_tokenizer = AutoTokenizer.from_pretrained(
    BEST_MODEL_DIR
)

print("불러온 모델:", BEST_MODEL_DIR)
print("클래스:", best_model.config.id2label)

In [ ]:
# 처음 보는 시험 데이터로 최종 분류 성능을 계산합니다.
# 학습·모델 선택에 사용하지 않은 시험 데이터로 최종 평가
test_result = evaluate(
    best_model,
    test_loader,
    DEVICE,
)

print(f"Accuracy : {test_result['accuracy']:.4f}")
print(f"Precision: {test_result['precision']:.4f}")
print(f"Recall   : {test_result['recall']:.4f}")
print(f"F1-score : {test_result['f1']:.4f}")
print(f"ROC-AUC  : {test_result['auc']:.4f}")

In [ ]:
# 셀 생략 가능
# 혼동행렬로 오탐·미탐 건수를 확인하고 클래스별 성능을 살펴봅니다.

# 어떤 클래스를 얼마나 잘못 분류했는지 추가로 확인
print("혼동 행렬:")
print(
    confusion_matrix(
        test_result["labels"],
        test_result["predictions"],
        labels=[0, 1],
    )
)

print("\n분류 보고서:")
print(
    classification_report(
        test_result["labels"],
        test_result["predictions"],
        target_names=["사람", "ChatGPT"],
        zero_division=0,
    )
)

In [ ]:
# 학습 기록과 최종 평가 지표를 Drive에 저장해 결과를 다시 확인할 수 있게 합니다.
# Epoch별 학습 기록 저장
history_df.to_csv(
    MODEL_DIR / "training_history.csv",
    index=False,
    encoding="utf-8-sig",
)

metrics = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "max_pairs": MAX_PAIRS,
    "max_length": MAX_LENGTH,
    "best_validation_f1": best_f1,
    "test_accuracy": test_result["accuracy"],
    "test_precision": test_result["precision"],
    "test_recall": test_result["recall"],
    "test_f1": test_result["f1"],
    "test_auc": test_result["auc"],
}

# 사용한 설정과 최종 평가 수치를 JSON으로 저장
with open(
    MODEL_DIR / "test_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("결과 저장 완료:", MODEL_DIR)

In [ ]:
# 새 글을 토큰화하고 ChatGPT 클래스의 출력 점수를 계산합니다.
@torch.no_grad()
def predict_score(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("검사할 글을 입력하세요.")

    # 원문이 최대 입력 길이를 넘었는지 확인
    token_count = len(
        best_tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )

    # 학습할 때와 같은 최대 길이로 변환
    encoded = best_tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    encoded = {
        key: value.to(DEVICE)
        for key, value in encoded.items()
    }

    best_model.eval()
    logits = best_model(**encoded).logits

    # ChatGPT 클래스(1)의 출력 점수를 0~1로 변환
    score = torch.softmax(
        logits,
        dim=1,
    )[0, 1].item()

    return {
        "score": score,
        "token_count": token_count,
        "truncated": token_count > MAX_LENGTH,
    }

In [ ]:
# 입력 글을 문장 단위로 나누어 참고용 구역 점수를 계산할 준비를 합니다.
def split_sentences(text):
    # 문장 끝의 마침표·물음표·느낌표 또는 줄바꿈을 기준으로 분리
    pieces = re.split(
        r"(?<=[.!?])\s+|\n+",
        text.strip(),
    )

    return [
        piece.strip()
        for piece in pieces
        if piece.strip()
    ]

In [ ]:
# 전체 글 점수와 문장 주변 문맥의 참고 점수를 화면에 표시합니다.
def show_detection(text):
    result = predict_score(text)
    score = result["score"]

    print(
        f"HC3-ko 기준 ChatGPT 의심 점수: "
        f"{score * 100:.1f}%"
    )

    # 두 클래스 중 어느 쪽 점수가 높은지 표시
    if score >= 0.5:
        print("판정: ChatGPT 답변에 가까움")
    else:
        print("판정: 사람 답변에 가까움")

    if result["truncated"]:
        print(
            f"전체 {result['token_count']}토큰 중 "
            f"앞 {MAX_LENGTH}토큰만 전체 점수에 사용했습니다."
        )

    sentences = split_sentences(text)

    if len(sentences) < 2:
        print("문장이 하나여서 구역별 표시는 생략합니다.")
        return

    print("\n의심 구역 참고 표시:")

    blocks = []

    for index, sentence in enumerate(sentences):
        # 현재 문장과 앞뒤 문장을 함께 검사
        start = max(0, index - 1)
        end = min(len(sentences), index + 2)

        context = " ".join(
            sentences[start:end]
        )

        local_score = predict_score(
            context
        )["score"]

        # 참고용 점수가 높은 문장을 붉은색으로 표시
        color = (
            "#ffe0e0"
            if local_score >= 0.5
            else "#e8f3e8"
        )

        safe_text = html.escape(sentence)

        blocks.append(
            f'<div style="background:{color}; '
            f'padding:8px; margin:5px 0; '
            f'border-radius:5px;">'
            f'{safe_text} '
            f'<b>({local_score * 100:.1f}%)</b>'
            f'</div>'
        )

    display(HTML("".join(blocks)))

In [ ]:
# 직접 작성한 글을 넣어 학습한 모델의 예측 결과를 확인합니다.
# 검사할 한국어 글을 이곳에 직접 입력
my_text = """
여기에 직접 검사할 한국어 글을 입력하세요.
문장을 여러 개 입력하면 의심 구역도 함께 표시됩니다.
"""

show_detection(my_text)